# LightMamba-ASL — Training on ASL Citizen Dataset
**GPU:** T4 (Free) or A100 (Pro)  
**Dataset:** ASL Citizen — 50 classes, 1870 videos  
**Model:** LightMamba-ASL (HMS-Mamba + MobileNetV3 + MediaPipe)

---
### Before running:
1. Upload `second_review.zip` (your project code) to Google Drive
2. Upload `dataset.zip` (containing `videos/` and `splits/`) to Google Drive
3. Runtime → Change runtime type → **T4 GPU**
4. Run cells top to bottom

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# List your Drive root to confirm files are there
os.listdir('/content/drive/MyDrive')

## Step 2 — Extract Project Code

In [ ]:
import zipfile, os, shutil

PROJECT_ZIP  = '/content/drive/MyDrive/second_review.zip'   # <-- adjust if needed
DATASET_ZIP  = '/content/drive/MyDrive/dataset.zip'         # <-- adjust if needed
PROJECT_DIR  = '/content/LightMamba'

# Extract project
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

print('Extracting project code...')
with zipfile.ZipFile(PROJECT_ZIP, 'r') as z:
    z.extractall(PROJECT_DIR)
print('Done.')

# List extracted contents
os.listdir(PROJECT_DIR)

## Step 3 — Extract Dataset

In [ ]:
DATASET_DIR = os.path.join(PROJECT_DIR, 'dataset')
os.makedirs(DATASET_DIR, exist_ok=True)

print('Extracting dataset (this may take a few minutes for 83k videos)...')
with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    z.extractall(DATASET_DIR)
print('Done.')

# Verify structure
print('Dataset contents:', os.listdir(DATASET_DIR))
videos_dir = os.path.join(DATASET_DIR, 'videos')
splits_dir = os.path.join(DATASET_DIR, 'splits')
print(f'Videos count: {len(os.listdir(videos_dir))}')
print(f'Splits: {os.listdir(splits_dir)}')

## Step 4 — Install Dependencies

In [ ]:
# Colab already has torch, torchvision — only install missing ones
!pip install -q mediapipe flask flask-cors tqdm opencv-python-headless
print('Dependencies installed.')

## Step 5 — Set Working Directory & Verify GPU

In [ ]:
import sys, os

# Find the actual project root (where backend/ folder lives)
# Handle cases where zip extracts into a subfolder
project_root = PROJECT_DIR
for item in os.listdir(PROJECT_DIR):
    candidate = os.path.join(PROJECT_DIR, item)
    if os.path.isdir(candidate) and 'backend' in os.listdir(candidate):
        project_root = candidate
        break

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Working directory: {os.getcwd()}')
print(f'Contents: {os.listdir(".")}')

import torch
print(f'\nCUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 6 — Prepare Dataset (Generate Processed Splits)

In [ ]:
# This reads dataset/splits/train|val|test.csv,
# filters to 50 classes, validates videos, saves to dataset/processed/splits/
!python -m backend.data.prepare_dataset

## Step 7 — Verify Model Loads Correctly

In [ ]:
import torch
from backend.models.lightmamba_asl import LightMambaASL
from backend.config import NUM_CLASSES, NUM_FRAMES, IMAGE_SIZE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LightMambaASL(pretrained=True, freeze_backbone=True).to(device)

# Dry run with dummy input
B, T, C, H, W = 2, NUM_FRAMES, 3, IMAGE_SIZE, IMAGE_SIZE
dummy_rgb  = torch.randn(B, T, C, H, W).to(device)
dummy_lm   = torch.randn(B, T, 450).to(device)   # landmarks + motion
dummy_mask = torch.ones(B, T, 3).to(device)

with torch.no_grad():
    out = model(dummy_rgb, dummy_lm, dummy_mask)

print(f'Model output shape : {out.shape}')   # expected: [2, 50]
print(f'NUM_CLASSES        : {NUM_CLASSES}')
print(f'Input RGB shape    : [{B}, {T}, {C}, {H}, {W}]')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters   : {total_params:,}')

## Step 8 — Start Training

In [ ]:
# Full training run — saves best_model.pth and last_model.pth to checkpoints/
# Stage 1 (epochs 1-20)  : MobileNetV3 frozen, only Mamba + fusion trains
# Stage 2 (epochs 21+)   : MobileNetV3 final layers unfrozen for fine-tuning
# Early stopping patience : 10 epochs
!python -m backend.training.train

## Step 9 — Evaluate on Test Set

In [ ]:
!python -m backend.evaluation.evaluate

## Step 10 — Download Checkpoint to Drive

In [ ]:
import shutil, os

SAVE_DIR = '/content/drive/MyDrive/LightMamba_outputs'
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy best model checkpoint
best_ckpt = os.path.join(project_root, 'checkpoints', 'best_model.pth')
if os.path.exists(best_ckpt):
    shutil.copy(best_ckpt, os.path.join(SAVE_DIR, 'best_model.pth'))
    print(f'best_model.pth saved to Drive: {SAVE_DIR}')
else:
    print('best_model.pth not found — check training completed successfully')

# Copy training plots
plots_src = os.path.join(project_root, 'outputs', 'plots')
if os.path.exists(plots_src):
    shutil.copytree(plots_src, os.path.join(SAVE_DIR, 'plots'), dirs_exist_ok=True)
    print('Training plots saved to Drive.')

# Copy confusion matrix
cm_src = os.path.join(project_root, 'outputs', 'confusion_matrix')
if os.path.exists(cm_src):
    shutil.copytree(cm_src, os.path.join(SAVE_DIR, 'confusion_matrix'), dirs_exist_ok=True)
    print('Confusion matrix saved to Drive.')

# Copy metrics
metrics_src = os.path.join(project_root, 'outputs', 'metrics')
if os.path.exists(metrics_src):
    shutil.copytree(metrics_src, os.path.join(SAVE_DIR, 'metrics'), dirs_exist_ok=True)
    print('Metrics saved to Drive.')

print('\nAll outputs saved to Google Drive!')
print(f'Location: {SAVE_DIR}')

## Step 11 — Show Training Curves
View loss and accuracy plots inline.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

plots_dir = os.path.join(project_root, 'outputs', 'plots')
for fname in ['training_curve_loss.png', 'training_curve_accuracy.png']:
    fpath = os.path.join(plots_dir, fname)
    if os.path.exists(fpath):
        img = mpimg.imread(fpath)
        plt.figure(figsize=(10, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(fname)
        plt.show()